# FAISS: Building a Vector Index (Indexing + Nearest Neighbor Search)

**Purpose:** Introduce FAISS as:

> **Indexing + nearest neighbor search** (NOT a magical database)

FAISS helps you:
- store vectors in an **index**
- run **nearest neighbor search** efficiently

But FAISS does **not**:
- store your original documents like a document DB
- manage complex metadata for you (you manage that separately)

In this notebook we will:
1. Explain what FAISS does.
2. Build a simple FAISS index (**IndexFlatL2** = exact search).
3. Add embeddings to the index.
4. Search: query → top-k indices + distances.
5. Map indices back to text.
6. Interpret distances (L2: smaller = closer).
7. Practical tips for real projects.


## 0) Setup

We’ll use:
- `sentence-transformers` to create embeddings
- `faiss` to build an index and search

Install notes:
- On many systems, `faiss-cpu` works well.
- If you're on GPU machines, there is also `faiss-gpu` (not used here).


In [1]:
# Install dependencies if needed (safe to re-run)
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    !uv add sentence-transformers

try:
    import faiss  # noqa: F401
except ImportError:
    !uv add faiss-cpu

Resolved 111 packages in 2.79s                                       
⠹ Preparing packages... (0/1)                                                   
⠹ Preparing packages... (0/1)-------------------     0 B/22.74 MiB           
⠹ Preparing packages... (0/1)------------------- 14.91 KiB/22.74 MiB         
⠹ Preparing packages... (0/1)------------------- 30.91 KiB/22.74 MiB         
⠹ Preparing packages... (0/1)------------------- 40.46 KiB/22.74 MiB         
⠹ Preparing packages... (0/1)------------------- 43.14 KiB/22.74 MiB         
⠹ Preparing packages... (0/1)------------------- 45.82 KiB/22.74 MiB         
⠹ Preparing packages... (0/1)------------------- 48.50 KiB/22.74 MiB         
⠹ Preparing packages... (0/1)------------------- 51.19 KiB/22.74 MiB         
⠹ Preparing packages... (0/1)------------------- 53.87 KiB/22.74 MiB         
⠹ Preparing packages... (0/1)------------------- 56.55 KiB/22.74 MiB         
⠹ Preparing packages... (0/1)------------------- 59.24 KiB/22.74 MiB 

In [2]:
import numpy as np
import pandas as pd

import faiss
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 120)

## 1) What FAISS does

FAISS is a library for **similarity search** / **nearest neighbor search** over vectors.

Think of it like:

- You already have vectors (embeddings).
- FAISS builds an **index** that makes searching fast.
- You query with a vector → FAISS returns **nearest vectors**.

### Key terms
- **Indexing**: adding vectors to a structure that supports fast search.
- **Searching**: finding nearest neighbors for a query vector.
- **Exact vs Approximate**:
  - Exact search returns true nearest neighbors (slower at huge scale).
  - Approximate search trades a bit of accuracy for speed (useful at millions/billions of vectors).


## 2) Index types (keep it simple)

We will use:

### ✅ `IndexFlatL2` (exact)
- Uses **L2 distance** (Euclidean distance)
- Exact brute-force search (fast enough for small datasets)

Optional mentions (no deep dive):
- **IVF** (Inverted File Index): approximate, faster at scale
- **HNSW** (graph-based): approximate, often strong recall/speed tradeoff


## 3) Load a tiny corpus + embed

We’ll keep a small corpus (like a toy RAG chunk store).


In [3]:
docs = [
    "Banks assess credit risk before approving loans.",
    "The stock market fell sharply after the earnings report.",
    "A loyalty program can increase customer retention.",
    "We ran SQL queries to validate the dataset.",
    "Machine learning models can recognize patterns in data.",
    "A typhoon is expected to make landfall tomorrow evening.",
    "Traffic on the expressway was heavy due to an accident.",
    "She bought gasoline before driving to the province.",
    "We should refactor the code to reduce technical debt.",
    "The chef prepared a spicy bowl of ramen.",
    "I enjoy reading novels on rainy afternoons.",
    "Coffee helps me stay focused during late-night work.",
    "Insurance companies price policies based on risk exposure.",
    "The athlete trained daily to improve endurance.",
    "Photosynthesis converts sunlight into chemical energy.",
]

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

# For L2-based FAISS indexes, it's common to use raw embeddings or normalized embeddings,
# depending on your chosen distance metric and retrieval design.
# We'll use raw embeddings here and interpret L2 distances directly.
emb = model.encode(docs, normalize_embeddings=False).astype("float32")

emb.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(15, 384)

### Inspect embedding shape

FAISS expects:
- a 2D array of shape **(N, dim)**
- dtype **float32**


In [4]:
print("Embedding matrix shape:", emb.shape)
print("dtype:", emb.dtype)
print("\nFirst document:")
print(" ", docs[0])
print("\nFirst embedding (first 5 values):")
print(emb[0][:5])

Embedding matrix shape: (15, 384)
dtype: float32

First document:
  Banks assess credit risk before approving loans.

First embedding (first 5 values):
[ 0.00692758 -0.0192121  -0.04428922  0.01720382  0.01129781]


## 4) Build an index (IndexFlatL2)

Steps:
1. Create the index with the embedding dimension.
2. Add vectors to the index.
3. Confirm how many vectors are stored.


In [5]:
dim = emb.shape[1]
index = faiss.IndexFlatL2(dim)  # exact L2 distance

index.add(emb)  # add vectors
print("Vectors in index:", index.ntotal)

Vectors in index: 15


## 5) Search: query → top-k indices + distances

FAISS returns:
- **D**: distances (L2 distance for IndexFlatL2)
- **I**: indices of nearest vectors

For L2 distance:
- **smaller distance = closer** (more similar in this metric)


In [6]:
query_text = "How do banks decide whether to approve a loan?"
query_vec = model.encode([query_text], normalize_embeddings=False).astype("float32")

k = 5
D, I = index.search(query_vec, k)

D, I

(array([[0.69395936, 1.7311864 , 1.7581398 , 1.8027816 , 1.863268  ]],
       dtype=float32),
 array([[0, 2, 4, 3, 8]]))

## 6) Map indices back to text

FAISS only stores vectors. **You must store your original documents elsewhere** (e.g., an array, database, files).

Here we keep:
- `docs` list as the source of truth for the raw text


In [7]:
results = []
for rank, (idx, dist) in enumerate(zip(I[0], D[0]), start=1):
    results.append({
        "rank": rank,
        "index": int(idx),
        "distance_L2": float(dist),
        "text": docs[int(idx)],
    })

results_df = pd.DataFrame(results)
results_df

,rank,index,distance_L2,text
0,1,0,0.693959,Banks assess credit risk before approving loans.
1,2,2,1.731186,A loyalty program can increase customer retention.
2,3,4,1.758140,Machine learning models can recognize patterns in data.
3,4,3,1.802782,We ran SQL queries to validate the dataset.
4,5,8,1.863268,We should refactor the code to reduce technical debt.


## 7) Interpret distances (L2)

With **IndexFlatL2**:
- distances are squared L2 by default in FAISS for IndexFlatL2 (implementation detail),
  but the key intuition still holds:
  - **smaller distance → closer neighbors**
  - **larger distance → farther neighbors**

Important: **Distance scale is model-dependent**
- Don’t compare distance magnitudes across different embedding models.
- Use distances mainly for ranking, thresholds, and diagnostics.


## 8) Practical tips (real-world patterns)

### Keep original docs separately
FAISS index contains vectors only. Keep your docs elsewhere:
- array/list (small demos)
- database / object storage (real systems)

### Store IDs + metadata separately
Common design:
- FAISS stores vectors in the same order you add them
- You maintain:
  - `doc_ids[i]` → the original ID for vector i
  - `metadata[doc_id]` → source, title, date, permissions, etc.

### Rebuild / update strategy
- `IndexFlatL2` is simple and exact, but can be slow at massive scale.
- For millions+ vectors, you might move to IVF or HNSW (approximate).

### Choose metric deliberately
- If you normalize embeddings and use `IndexFlatIP` (inner product),
  you effectively do cosine similarity.
- Here we used L2 for simplicity and clear “distance” intuition.


## Outputs checklist

- ✅ Distances and top-k results printed clearly
- ✅ Indices mapped back to original text
- ✅ Interpretation: smaller distance = closer (for L2)
- ✅ Practical tips for storing docs + metadata


## Mini reflection (student prompt)

1. What does FAISS store? What does it *not* store?
2. If your system returns index `42`, how do you retrieve the original text?
3. If you changed from L2 to cosine retrieval, what would you need to change:
   - metric choice
   - embedding normalization
   - index type
